# Model Evaluation & Hyperparameter Tuning with scikit-learn

Hands-on practice with three core ML workflow skills:

1. **Cross-validation**: hold-out splits vs. K-fold, and the accuracy/runtime trade-off
2. **Classification metrics**: accuracy, precision, recall and F1 against a naive baseline
3. **Hyperparameter search**: `GridSearchCV` and `RandomizedSearchCV` with different scoring metrics

## 1. Cross-Validation Strategies

Using the **California Housing** dataset (`datasets.fetch_california_housing()`), compare three regressors (`KNeighborsRegressor`, `LinearRegression`, `DecisionTreeRegressor`) for predicting the median house value of a block group.

The same models are evaluated with:
- a train/test split (10% hold-out)
- K-fold cross-validation with K = 10, 100 and 1000

Both the R² score and the runtime are recorded, to show how the choice of evaluation strategy affects reliability and cost.

In [1]:
import numpy as np
import time
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

housing = datasets.fetch_california_housing()
scaled_data = StandardScaler().fit_transform(housing.data)
target = housing.target

models = {"KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42)}

print("Train/Test Split (10% holdout)")

X_train, X_test, y_train, y_test = train_test_split(scaled_data, target, test_size=0.10, random_state=42)

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    elapsed = time.time() - start
    print(f"{name}: R^2 = {score:.4f}, Time = {elapsed:.4f}s")

for k in [10, 100, 1000]:
    print(f"\n K-Fold Cross Validation (K={k})")
    kfold = KFold(n_splits=k, shuffle=True, random_state=42)

    for name, model in models.items():
        cv_scores = np.zeros(k)
        start = time.time()

        for counter, (train_idx, test_idx) in enumerate(kfold.split(scaled_data, target)):
            X_train_kfold = scaled_data[train_idx, ]
            y_train_kfold = target[train_idx]
            X_test_kfold = scaled_data[test_idx, ]
            y_test_kfold = target[test_idx]

            model.fit(X_train_kfold, y_train_kfold)
            cv_scores[counter] = model.score(X_test_kfold, y_test_kfold)

        elapsed = time.time() - start
        print(f"{name}: Mean R^2 = {cv_scores.mean():.4f}, Time = {elapsed:.4f}s")


Train/Test Split (10% holdout)
KNeighborsRegressor: R^2 = 0.6718, Time = 0.2985s
LinearRegression: R^2 = 0.5808, Time = 0.0798s
DecisionTreeRegressor: R^2 = 0.6502, Time = 0.3326s

 K-Fold Cross Validation (K=10)
KNeighborsRegressor: Mean R^2 = 0.6912, Time = 2.9058s
LinearRegression: Mean R^2 = 0.6001, Time = 0.0833s
DecisionTreeRegressor: Mean R^2 = 0.6105, Time = 3.3235s

 K-Fold Cross Validation (K=100)
KNeighborsRegressor: Mean R^2 = 0.6905, Time = 7.8508s
LinearRegression: Mean R^2 = 0.6013, Time = 0.9031s
DecisionTreeRegressor: Mean R^2 = 0.6074, Time = 36.4308s

 K-Fold Cross Validation (K=1000)
KNeighborsRegressor: Mean R^2 = 0.6580, Time = 54.1552s
LinearRegression: Mean R^2 = 0.5646, Time = 8.8081s
DecisionTreeRegressor: Mean R^2 = 0.5559, Time = 397.4616s


## 2. Classification Metrics vs. a Naive Baseline

Using the **Breast Cancer Wisconsin (Diagnostic)** dataset ([`load_breast_cancer`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)), a `RandomForestClassifier` predicts whether a tumour is malignant (0) or benign (1).

It is evaluated with stratified 10-fold cross-validation on:
- Accuracy
- Precision
- Recall
- F1-score

For comparison, a `DummyClassifier(strategy='most_frequent')` always predicts the majority class (1). Its scores show why accuracy alone can be misleading on imbalanced data: the baseline gets a perfect recall of 1.0 and still has no predictive value.

In [2]:
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

breast_cancer = datasets.load_breast_cancer()
scaled_data = StandardScaler().fit_transform(breast_cancer.data)
target = breast_cancer.target

K = 10
skfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

rf = RandomForestClassifier(random_state=42)

cv_accuracy = np.zeros(K)
cv_precision = np.zeros(K)
cv_recall = np.zeros(K)
cv_f1 = np.zeros(K)

for counter, (train_idx, test_idx) in enumerate(skfold.split(scaled_data, target)):
    X_train_kfold = scaled_data[train_idx, ]
    y_train_kfold = target[train_idx, ]
    X_test_kfold = scaled_data[test_idx, ]
    y_test_kfold = target[test_idx, ]

    rf.fit(X_train_kfold, y_train_kfold)
    preds = rf.predict(X_test_kfold)

    cv_accuracy[counter] = accuracy_score(y_test_kfold, preds)
    cv_precision[counter] = precision_score(y_test_kfold, preds)
    cv_recall[counter] = recall_score(y_test_kfold, preds)
    cv_f1[counter] = f1_score(y_test_kfold, preds)

print("RandomForestClassifier")
print("accuracy:", cv_accuracy)
print("Mean of accuracy:", cv_accuracy.mean())

print("precision:", cv_precision)
print("Mean of precision:", cv_precision.mean())

print("recall:", cv_recall)
print("Mean of recall:", cv_recall.mean())

print("f1:", cv_f1)
print("Mean of f1:", cv_f1.mean())


dc = DummyClassifier(strategy='most_frequent')

dc_accuracy = np.zeros(K)
dc_precision = np.zeros(K)
dc_recall = np.zeros(K)
dc_f1 = np.zeros(K)

for counter, (train_idx, test_idx) in enumerate(skfold.split(scaled_data, target)):
    X_train_kfold = scaled_data[train_idx, ]
    y_train_kfold = target[train_idx, ]
    X_test_kfold = scaled_data[test_idx, ]
    y_test_kfold = target[test_idx, ]

    dc.fit(X_train_kfold, y_train_kfold)
    preds = dc.predict(X_test_kfold)

    dc_accuracy[counter] = accuracy_score(y_test_kfold, preds)
    dc_precision[counter] = precision_score(y_test_kfold, preds)
    dc_recall[counter] = recall_score(y_test_kfold, preds)
    dc_f1[counter] = f1_score(y_test_kfold, preds)

print("\n DummyClassifier (baseline)")
print("accuracy:", dc_accuracy)
print("Mean of accuracy:", dc_accuracy.mean())

print("precision:", dc_precision)
print("Mean of precision:", dc_precision.mean())

print("recall:", dc_recall)
print("Mean of recall:", dc_recall.mean())

print("f1:", dc_f1)
print("Mean of f1:", dc_f1.mean())


RandomForestClassifier
accuracy: [0.94736842 0.98245614 0.96491228 0.9122807  0.94736842 0.96491228
 0.96491228 0.94736842 0.92982456 1.        ]
Mean of accuracy: 0.956140350877193
precision: [0.97058824 1.         0.97222222 0.8974359  0.94594595 0.94736842
 0.97222222 1.         0.92105263 1.        ]
Mean of precision: 0.9626835575751984
recall: [0.94285714 0.97142857 0.97222222 0.97222222 0.97222222 1.
 0.97222222 0.91666667 0.97222222 1.        ]
Mean of recall: 0.9692063492063492
f1: [0.95652174 0.98550725 0.97222222 0.93333333 0.95890411 0.97297297
 0.97222222 0.95652174 0.94594595 1.        ]
Mean of f1: 0.9654151530923419

 DummyClassifier (baseline)
accuracy: [0.61403509 0.61403509 0.63157895 0.63157895 0.63157895 0.63157895
 0.63157895 0.63157895 0.63157895 0.625     ]
Mean of accuracy: 0.6274122807017544
precision: [0.61403509 0.61403509 0.63157895 0.63157895 0.63157895 0.63157895
 0.63157895 0.63157895 0.63157895 0.625     ]
Mean of precision: 0.6274122807017544
recall: [

## 3. Hyperparameter Tuning: Grid Search & Randomized Search

**a) Random Forest regressor on California Housing (`GridSearchCV`)**
1. Fix the random seed (`np.random.seed(42)`)
2. Split into train/test sets (`test_size=0.2`)
3. Standardise the features using statistics from the training set only
4. Use a `RandomForestRegressor` with a shuffled 5-fold `KFold`
5. Search `{'n_estimators': [10, 50, 100], 'min_samples_leaf': [2, 10]}` with `scoring='r2'` and `n_jobs=-1`
6. Report the best hyperparameter combination

**b) Distance-weighted KNN classifier on Iris (`GridSearchCV`)**
- metric ∈ {euclidean, cosine, manhattan, minkowski}
- K ∈ {1, 3, 5, 10, 50}
- Compared under both `accuracy` and `f1_macro` scoring, to see whether the best combination changes

**c) Decision tree classifier on Breast Cancer (`RandomizedSearchCV`)**
- max_depth ∈ [1..50], min_samples_split ∈ [2..32], min_samples_leaf ∈ [1..50], max_leaf_nodes ∈ [2..128]
- An exhaustive grid would be far too large, so `RandomizedSearchCV` samples `n_iter=1000` combinations instead
- Compared under both `accuracy` and `f1` scoring

In [3]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

np.random.seed(42)

housing = datasets.fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestRegressor(random_state=42)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

grids = {'n_estimators': [10, 50, 100], 'min_samples_leaf': [2, 10]}

gscv = GridSearchCV(rf, grids, cv=kfold, n_jobs=-1, scoring='r2')
gscv.fit(X_train_scaled, y_train)

print("California Housing - RandomForestRegressor")
print(gscv.best_params_)
print(gscv.best_score_)

score_best_estimator = gscv.score(X_test_scaled, y_test)
print("Test R^2:", score_best_estimator)

np.random.seed(42)

iris = datasets.load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(weights='distance')
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

grids = {'metric': ['euclidean', 'cosine', 'manhattan', 'minkowski'],'n_neighbors': [1, 3, 5, 10, 50]}

gscv_acc = GridSearchCV(knn, grids, cv=kfold, n_jobs=-1, scoring='accuracy')
gscv_acc.fit(X_train_scaled, y_train)

print("\nIris - KNN (scoring = accuracy)")
print(gscv_acc.best_params_)
print(gscv_acc.best_score_)
print("Test accuracy:", gscv_acc.score(X_test_scaled, y_test))

gscv_f1 = GridSearchCV(knn, grids, cv=kfold, n_jobs=-1, scoring='f1_macro')
gscv_f1.fit(X_train_scaled, y_train)

print("\nIris - KNN (scoring = f1_macro)")
print(gscv_f1.best_params_)
print(gscv_f1.best_score_)
print("Test f1_macro:", gscv_f1.score(X_test_scaled, y_test))

print("\nSame best combination for both metrics?", gscv_acc.best_params_ == gscv_f1.best_params_)

np.random.seed(42)

breast_cancer = datasets.load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(breast_cancer.data, breast_cancer.target, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

dt = DecisionTreeClassifier(random_state=42)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

grids = {'max_depth': list(range(1, 51)),'min_samples_split': list(range(2, 33)),'min_samples_leaf': list(range(1, 51)),'max_leaf_nodes': list(range(2, 129))}

rscv_acc = RandomizedSearchCV(dt, grids, n_iter=1000, cv=kfold, n_jobs=-1,scoring='accuracy', random_state=42)
rscv_acc.fit(X_train_scaled, y_train)

print("\nBreast Cancer - Decision Tree (scoring = accuracy)")
print(rscv_acc.best_params_)
print(rscv_acc.best_score_)
print("Test accuracy:", rscv_acc.score(X_test_scaled, y_test))

rscv_f1 = RandomizedSearchCV(dt, grids, n_iter=1000, cv=kfold, n_jobs=-1,scoring='f1', random_state=42)
rscv_f1.fit(X_train_scaled, y_train)

print("\nBreast Cancer - Decision Tree (scoring = f1)")
print(rscv_f1.best_params_)
print(rscv_f1.best_score_)
print("Test f1:", rscv_f1.score(X_test_scaled, y_test))

print("\nSame best combination for both metrics?", rscv_acc.best_params_ == rscv_f1.best_params_)

California Housing - RandomForestRegressor
{'min_samples_leaf': 2, 'n_estimators': 100}
0.8046916345150839
Test R^2: 0.8060211966170572

Iris - KNN (scoring = accuracy)
{'metric': 'euclidean', 'n_neighbors': 5}
0.95
Test accuracy: 1.0

Iris - KNN (scoring = f1_macro)
{'metric': 'euclidean', 'n_neighbors': 5}
0.9436243386243387
Test f1_macro: 1.0

Same best combination for both metrics? True

Breast Cancer - Decision Tree (scoring = accuracy)
{'min_samples_split': 8, 'min_samples_leaf': 17, 'max_leaf_nodes': 80, 'max_depth': 38}
0.9208791208791208
Test accuracy: 0.9473684210526315

Breast Cancer - Decision Tree (scoring = f1)
{'min_samples_split': 8, 'min_samples_leaf': 17, 'max_leaf_nodes': 80, 'max_depth': 38}
0.9360704643367181
Test f1: 0.9577464788732394

Same best combination for both metrics? True
